In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.5 FlashAttention

The attention we built materializes the full `T x T` score matrix. **FlashAttention**
computes the same result in tiles without ever storing that matrix, saving memory
and time. It is the same math, so the outputs match, we teach with the explicit
version and ship the fused one.

In [ ]:
from torch.nn import functional as F
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 6, 8)
k = torch.randn(1, 4, 6, 8)
v = torch.randn(1, 4, 6, 8)
ours, _ = scaled_dot_product_attention(q, k, v, causal=True)
fused = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # fused kernel
print("max difference:", (ours - fused).abs().max().item())

The two agree to floating-point tolerance. In production you call the fused path;
in this course we keep the explicit one because you can read it.

In [ ]:
assert torch.allclose(ours, fused, atol=1e-5)